# Principal Axes and Canonical Matrix Transformations

This notebook demonstrates principal axis analysis and canonical matrix transformations
for topology alignment using **topologic_fast**.

## Concept

Principal Component Analysis (PCA) can be applied to 3D geometry to find the main axes
of orientation. The **canonical matrix** transforms a topology to a standard orientation
aligned with the coordinate axes, which is useful for:

- Comparing shapes regardless of their position/orientation
- Standardizing BIM elements for classification
- Shape matching and recognition
- Geometric hashing

## Note on topologicpy vs topologic_fast

The original topologicpy has `Topology.PrincipalAxes()` and `Topology.CanonicalMatrix()`.
In topologic_fast, we demonstrate PCA-based principal axis calculation using numpy
and show how to apply transformations using the Matrix utilities.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np
import math

## Helper Functions

First, let's create helper functions for visualization and principal axis calculation.

In [ ]:
def get_vertices_as_array(topology):
    """
    Extract vertices from any topology as a numpy array.
    
    Works with Cell, CellComplex, Face, Wire, etc.
    """
    vertices = topology.Vertices()
    coords = [v.Coordinates() for v in vertices]
    return np.array(coords)


def compute_centroid(points):
    """Compute the centroid of a point cloud."""
    return np.mean(points, axis=0)


def compute_principal_axes(points):
    """
    Compute principal axes using PCA.
    
    Returns:
        centroid: The center of the point cloud
        axes: 3x3 matrix where rows are principal axes (sorted by variance)
        variances: Variance along each principal axis
    """
    centroid = compute_centroid(points)
    centered = points - centroid
    
    # Compute covariance matrix
    cov = np.cov(centered.T)
    
    # Compute eigenvalues and eigenvectors
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # Sort by eigenvalue (descending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Ensure right-handed coordinate system
    if np.linalg.det(eigenvectors) < 0:
        eigenvectors[:, 2] *= -1
    
    return centroid, eigenvectors.T, eigenvalues


def compute_canonical_matrix(points, normalize=True):
    """
    Compute the canonical transformation matrix.
    
    This matrix transforms the geometry to align with coordinate axes,
    centered at the origin.
    
    Parameters:
        points: Nx3 array of point coordinates
        normalize: If True, also scale to unit bounding box
    
    Returns:
        4x4 transformation matrix
    """
    centroid, axes, variances = compute_principal_axes(points)
    
    # Create rotation matrix (axes are rows, so we transpose)
    rotation = axes  # Each row is a principal axis
    
    # Create 4x4 transformation matrix
    # First translate to origin, then rotate
    matrix = np.eye(4)
    matrix[:3, :3] = rotation
    matrix[:3, 3] = -rotation @ centroid
    
    if normalize:
        # Transform points to get bounding box
        transformed = (rotation @ (points - centroid).T).T
        mins = transformed.min(axis=0)
        maxs = transformed.max(axis=0)
        extents = maxs - mins
        
        # Scale to unit cube (avoid division by zero)
        scale = np.where(extents > 1e-10, 1.0 / extents, 1.0)
        
        # Create scaling matrix
        scale_matrix = np.diag([scale[0], scale[1], scale[2], 1.0])
        
        # Center the scaled result
        center_offset = -(mins * scale + maxs * scale) / 2
        translate_matrix = np.eye(4)
        translate_matrix[:3, 3] = center_offset
        
        matrix = translate_matrix @ scale_matrix @ matrix
    
    return matrix


def invert_matrix_4x4(matrix):
    """Invert a 4x4 transformation matrix."""
    return np.linalg.inv(matrix)


def transform_points(points, matrix):
    """Transform points by a 4x4 matrix."""
    # Add homogeneous coordinate
    ones = np.ones((points.shape[0], 1))
    homogeneous = np.hstack([points, ones])
    
    # Transform
    transformed = (matrix @ homogeneous.T).T
    
    # Return 3D coordinates
    return transformed[:, :3]

In [ ]:
def visualize_3d_with_axes(points_list, axes_list=None, centroids=None, 
                          colors=None, names=None, title='3D Visualization',
                          show_coord_axes=True, axis_size=2.0):
    """
    Visualize point clouds with optional principal axes.
    
    Parameters:
        points_list: List of Nx3 arrays
        axes_list: List of 3x3 arrays (principal axes for each point cloud)
        centroids: List of centroids
        colors: List of colors
        names: List of names
    """
    fig = go.Figure()
    
    if colors is None:
        colors = ['blue', 'red', 'green', 'orange', 'purple'] * len(points_list)
    if names is None:
        names = [f'Object {i+1}' for i in range(len(points_list))]
    
    # Plot point clouds
    for i, (points, color, name) in enumerate(zip(points_list, colors, names)):
        fig.add_trace(go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode='markers',
            marker=dict(size=5, color=color, opacity=0.7),
            name=name
        ))
    
    # Plot principal axes
    if axes_list is not None and centroids is not None:
        axis_colors = ['red', 'green', 'blue']  # X, Y, Z colors
        axis_names = ['X-axis', 'Y-axis', 'Z-axis']
        
        for obj_idx, (axes, centroid, color) in enumerate(zip(axes_list, centroids, colors)):
            for ax_idx, (axis, ax_color, ax_name) in enumerate(zip(axes, axis_colors, axis_names)):
                end = centroid + axis * axis_size
                fig.add_trace(go.Scatter3d(
                    x=[centroid[0], end[0]],
                    y=[centroid[1], end[1]],
                    z=[centroid[2], end[2]],
                    mode='lines',
                    line=dict(color=ax_color, width=6),
                    name=f'{names[obj_idx]} {ax_name}',
                    showlegend=(obj_idx == 0)
                ))
    
    # Add coordinate axes
    if show_coord_axes:
        origin = [0, 0, 0]
        for i, (color, name) in enumerate(zip(['red', 'green', 'blue'], ['X', 'Y', 'Z'])):
            end = [0, 0, 0]
            end[i] = axis_size * 1.5
            fig.add_trace(go.Scatter3d(
                x=[origin[0], end[0]],
                y=[origin[1], end[1]],
                z=[origin[2], end[2]],
                mode='lines+text',
                line=dict(color=color, width=3, dash='dash'),
                text=['', name],
                textposition='top center',
                name=f'Coord {name}',
                showlegend=False
            ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        width=900,
        height=700
    )
    
    return fig

## Create Test Objects

Let's create two prism objects with different positions, orientations, and scales.

In [ ]:
# Create a base rectangular prism (non-cube for clear axis identification)
# Width=1, Length=2, Height=3 makes it easy to see the principal axes
base_cell = tf.Cell.Box(0, 0, 0, 1.0, 2.0, 3.0)

print(f"Base Cell Properties:")
print(f"  Volume: {base_cell.Volume():.2f}")
print(f"  Surface Area: {base_cell.Area():.2f}")
print(f"  Vertices: {len(base_cell.Vertices())}")
print(f"  Faces: {len(base_cell.Faces())}")

In [ ]:
# Get vertices of the base cell
base_points = get_vertices_as_array(base_cell)

# Create Object 1: Translate and rotate the base cell
# Translate by (6, 4, 5) and rotate around axis [1, 1, 1] by 32.5 degrees
def rotate_points(points, axis, angle_deg):
    """Rotate points around an axis by angle (in degrees)."""
    axis = np.array(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)
    angle = np.radians(angle_deg)
    
    # Rodrigues' rotation formula
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    
    K = np.array([
        [0, -axis[2], axis[1]],
        [axis[2], 0, -axis[0]],
        [-axis[1], axis[0], 0]
    ])
    
    R = np.eye(3) + sin_a * K + (1 - cos_a) * (K @ K)
    
    centroid = np.mean(points, axis=0)
    centered = points - centroid
    rotated = (R @ centered.T).T
    return rotated + centroid


def scale_points(points, sx, sy, sz):
    """Scale points from their centroid."""
    centroid = np.mean(points, axis=0)
    centered = points - centroid
    scaled = centered * np.array([sx, sy, sz])
    return scaled + centroid


# Object 1: Translate and rotate
obj1_points = base_points + np.array([6, 4, 5])
obj1_points = rotate_points(obj1_points, [1, 1, 1], 32.5)
obj1_points = scale_points(obj1_points, 1.2, 1.2, 1.2)

# Object 2: Different translation and rotation
obj2_points = base_points + np.array([5.4, -2, 6])
obj2_points = rotate_points(obj2_points, [1, 1, 0.6], 90)
obj2_points = scale_points(obj2_points, 2, 2, 2)

print(f"Object 1: Translated by (6, 4, 5), Rotated by 32.5 deg, Scaled by 1.2")
print(f"  Centroid: {compute_centroid(obj1_points)}")

print(f"\nObject 2: Translated by (5.4, -2, 6), Rotated by 90 deg, Scaled by 2.0")
print(f"  Centroid: {compute_centroid(obj2_points)}")

In [ ]:
# Visualize the two objects
fig = visualize_3d_with_axes(
    [obj1_points, obj2_points],
    colors=['blue', 'orange'],
    names=['Object 1 (Translated, Rotated, Scaled)', 'Object 2 (Different Transform)'],
    title='Two Randomly Positioned Objects',
    show_coord_axes=True
)
fig.show()

## Compute Principal Axes

Now let's compute the principal axes for each object.

In [ ]:
# Compute principal axes for both objects
centroid1, axes1, var1 = compute_principal_axes(obj1_points)
centroid2, axes2, var2 = compute_principal_axes(obj2_points)

print("Object 1 Principal Axes:")
print(f"  Centroid: [{centroid1[0]:.3f}, {centroid1[1]:.3f}, {centroid1[2]:.3f}]")
print(f"  X-axis (primary): [{axes1[0, 0]:.3f}, {axes1[0, 1]:.3f}, {axes1[0, 2]:.3f}]")
print(f"  Y-axis (secondary): [{axes1[1, 0]:.3f}, {axes1[1, 1]:.3f}, {axes1[1, 2]:.3f}]")
print(f"  Z-axis (tertiary): [{axes1[2, 0]:.3f}, {axes1[2, 1]:.3f}, {axes1[2, 2]:.3f}]")
print(f"  Variances: [{var1[0]:.3f}, {var1[1]:.3f}, {var1[2]:.3f}]")

print("\nObject 2 Principal Axes:")
print(f"  Centroid: [{centroid2[0]:.3f}, {centroid2[1]:.3f}, {centroid2[2]:.3f}]")
print(f"  X-axis (primary): [{axes2[0, 0]:.3f}, {axes2[0, 1]:.3f}, {axes2[0, 2]:.3f}]")
print(f"  Y-axis (secondary): [{axes2[1, 0]:.3f}, {axes2[1, 1]:.3f}, {axes2[1, 2]:.3f}]")
print(f"  Z-axis (tertiary): [{axes2[2, 0]:.3f}, {axes2[2, 1]:.3f}, {axes2[2, 2]:.3f}]")
print(f"  Variances: [{var2[0]:.3f}, {var2[1]:.3f}, {var2[2]:.3f}]")

In [ ]:
# Visualize objects with their principal axes
fig = visualize_3d_with_axes(
    [obj1_points, obj2_points],
    axes_list=[axes1, axes2],
    centroids=[centroid1, centroid2],
    colors=['blue', 'orange'],
    names=['Object 1', 'Object 2'],
    title='Objects with Principal Axes',
    show_coord_axes=True,
    axis_size=3.0
)
fig.show()

## Canonical Transformation

The canonical transformation aligns the object's principal axes with the coordinate axes
and centers it at the origin. This creates a standard representation of the shape.

In [ ]:
# Compute canonical matrices
matrix1 = compute_canonical_matrix(obj1_points, normalize=True)
matrix2 = compute_canonical_matrix(obj2_points, normalize=True)

print("Canonical Matrix for Object 1:")
print(matrix1.round(4))

print("\nCanonical Matrix for Object 2:")
print(matrix2.round(4))

In [ ]:
# Transform objects to canonical form
can_obj1_points = transform_points(obj1_points, matrix1)
can_obj2_points = transform_points(obj2_points, matrix2)

print("Canonical Object 1:")
print(f"  Centroid: {compute_centroid(can_obj1_points).round(6)}")
print(f"  Bounding Box: min={can_obj1_points.min(axis=0).round(3)}, max={can_obj1_points.max(axis=0).round(3)}")

print("\nCanonical Object 2:")
print(f"  Centroid: {compute_centroid(can_obj2_points).round(6)}")
print(f"  Bounding Box: min={can_obj2_points.min(axis=0).round(3)}, max={can_obj2_points.max(axis=0).round(3)}")

In [ ]:
# Visualize canonical forms alongside originals
fig = visualize_3d_with_axes(
    [obj1_points, obj2_points, can_obj1_points, can_obj2_points],
    colors=['blue', 'orange', 'lightblue', 'gold'],
    names=['Object 1 (Original)', 'Object 2 (Original)', 
           'Object 1 (Canonical)', 'Object 2 (Canonical)'],
    title='Original Objects and Canonical Forms',
    show_coord_axes=True,
    axis_size=2.0
)
fig.show()

In [ ]:
# Visualize just the canonical forms (they should overlap!)
fig = visualize_3d_with_axes(
    [can_obj1_points, can_obj2_points],
    colors=['blue', 'orange'],
    names=['Canonical Object 1', 'Canonical Object 2'],
    title='Canonical Forms (Should Overlap for Same Shape)',
    show_coord_axes=True,
    axis_size=0.7
)
fig.show()

## Transform to Match Another Object

We can use the inverse of the canonical matrix to transform one object to match
the position and orientation of another object.

In [ ]:
# Get the inverse of Object 1's canonical matrix
matrix1_inv = invert_matrix_4x4(matrix1)

print("Inverse of Canonical Matrix 1:")
print(matrix1_inv.round(4))

In [ ]:
# Transform canonical Object 2 to match Object 1's position/orientation
# This is: Original_1_coords = Inv(M1) @ Can_2_coords
obj2_matched_to_obj1 = transform_points(can_obj2_points, matrix1_inv)

print("Object 2 Transformed to Match Object 1:")
print(f"  Centroid: {compute_centroid(obj2_matched_to_obj1).round(3)}")
print(f"  Object 1 Centroid: {compute_centroid(obj1_points).round(3)}")

In [ ]:
# Visualize the matching - Object 1 and the transformed Object 2 should overlap
fig = visualize_3d_with_axes(
    [obj1_points, obj2_matched_to_obj1],
    colors=['blue', 'red'],
    names=['Object 1 (Original)', 'Object 2 (Transformed to Match Object 1)'],
    title='Object 2 Transformed to Match Object 1 (Should Overlap)',
    show_coord_axes=True,
    axis_size=3.0
)
fig.show()

## Using topologic_fast Matrix Operations

topologic_fast provides Matrix and Topology transformation utilities that can be used
for similar operations.

In [ ]:
# Demonstrate topologic_fast Matrix operations
identity = tf.Matrix.Identity()
print("Identity Matrix:")
for row in identity:
    print(f"  {row}")

# Create rotation matrix
rotation = tf.Matrix.ByRotation(rx=0, ry=0, rz=45)  # 45 degrees around Z
print("\nRotation Matrix (45 deg around Z):")
for row in rotation:
    print(f"  [{row[0]:.4f}, {row[1]:.4f}, {row[2]:.4f}, {row[3]:.4f}]")

# Create translation matrix
translation = tf.Matrix.ByTranslation(tx=5, ty=3, tz=1)
print("\nTranslation Matrix (5, 3, 1):")
for row in translation:
    print(f"  {row}")

In [ ]:
# Multiply matrices (combine transformations)
combined = tf.Matrix.Multiply(translation, rotation)
print("Combined Matrix (Translation * Rotation):")
for row in combined:
    print(f"  [{row[0]:.4f}, {row[1]:.4f}, {row[2]:.4f}, {row[3]:.4f}]")

In [ ]:
# Demonstrate using topologic_fast transformations on a Face
face = tf.Face.Rectangle(width=2, length=1)
original_com = face.CenterOfMass()
print(f"Original Face Center of Mass: {original_com}")

# Translate the face
translated_face = tf.Topology.TranslateFace(face, x=3, y=2, z=0)
translated_com = translated_face.CenterOfMass()
print(f"Translated Face Center of Mass: {translated_com}")

# Rotate the face
rotated_face = tf.Topology.RotateFace(face, axis=[0, 0, 1], angle=45)
rotated_com = rotated_face.CenterOfMass()
print(f"Rotated Face Center of Mass: {rotated_com}")

In [ ]:
# Transform a Face using a custom matrix
# NOTE: Topology.TransformFace is available in topologic_fast
custom_matrix = [
    [1.5, 0, 0, 2],   # Scale X by 1.5, translate by 2
    [0, 1.5, 0, 3],   # Scale Y by 1.5, translate by 3
    [0, 0, 1, 0],     # Z unchanged
    [0, 0, 0, 1]      # Homogeneous
]

transformed_face = tf.Topology.TransformFace(face, custom_matrix)
transformed_com = transformed_face.CenterOfMass()
print(f"Custom Transform Face Center of Mass: {transformed_com}")
print(f"Expected: (2 + 0*1.5, 3 + 0*1.5, 0) = (2, 3, 0)")

## Vector Utilities

topologic_fast also provides Vector utilities for geometric calculations.

In [ ]:
# Vector operations
v1 = [1, 0, 0]
v2 = [1, 1, 0]

# Angle between vectors
angle = tf.Vector.Angle(v1, v2)
print(f"Angle between {v1} and {v2}: {angle:.2f} degrees")

# Cross product
cross = tf.Vector.Cross(v1, v2)
print(f"Cross product: {cross}")

# Dot product
dot = tf.Vector.Dot(v1, v2)
print(f"Dot product: {dot}")

# Magnitude
mag = tf.Vector.Magnitude(v2)
print(f"Magnitude of {v2}: {mag:.4f}")

# Normalize
normalized = tf.Vector.Normalize(v2)
print(f"Normalized {v2}: {[round(x, 4) for x in normalized]}")

# Compass directions
print(f"\nCardinal directions:")
print(f"  North: {tf.Vector.North()}")
print(f"  East:  {tf.Vector.East()}")
print(f"  Up:    {tf.Vector.Up()}")

## Summary

This notebook demonstrated:

1. **Principal Axis Analysis**: Using PCA to find the main axes of orientation

2. **Canonical Transformation**: Creating a standard representation aligned with coordinate axes

3. **Shape Matching**: Using inverse transforms to align one object with another

4. **Matrix Operations**: Using `tf.Matrix` for rotation, translation, scaling

5. **Topology Transforms**: Using `tf.Topology.TranslateFace`, `RotateFace`, `TransformFace`

6. **Vector Utilities**: Using `tf.Vector` for geometric calculations

### API Differences from topologicpy

| topologicpy | topologic_fast | Notes |
|------------|----------------|-------|
| `Topology.PrincipalAxes()` | Not available | Use numpy PCA as shown above |
| `Topology.CanonicalMatrix()` | Not available | Use numpy implementation as shown above |
| `Matrix.Invert()` | Not available | Use `numpy.linalg.inv()` |
| `Topology.Transform()` | `tf.Topology.TransformFace/Cell()` | Available for Face and Cell |
| `Matrix.ByRotation()` | `tf.Matrix.ByRotation()` | Same API |

### Applications

- BIM element classification
- Shape recognition and matching
- Geometric hashing for collision detection
- Standardizing geometry for analysis
- View-independent shape comparison